# 04 — Neural Network from Scratch

## Why Do We Need Layers?

A single perceptron can only draw a straight line.  XOR isn't straight.  
But if we **stack neurons in layers**, each layer can reshape the decision  
boundary — and together they can learn any function.

This notebook builds a tiny neural network with:
- **Input layer**: 2 inputs (A and B)
- **Hidden layer**: 3 neurons (they learn useful internal representations)
- **Output layer**: 1 neuron (predicts XOR)

We write every calculation by hand so you can see exactly what PyTorch  
(and TensorFlow, etc.) are doing under the hood.

## The XOR Dataset

| A | B | A XOR B |
|---|---|---------|
| 0 | 0 |    0    |
| 0 | 1 |    1    |
| 1 | 0 |    1    |
| 1 | 1 |    0    |

The '0,0→0' and '1,1→0' are on one side; '0,1→1' and '1,0→1' on the other.  
They're arranged in an X pattern — no straight line can separate them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_d(x):
    """Derivative of sigmoid — used in backprop."""
    s = sigmoid(x)
    return s * (1 - s)

# 4 training examples, 2 features each
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

# Target outputs (column vector for matrix math convenience)
y = np.array([[0],
              [1],
              [1],
              [0]])

## Network Architecture

```
  Input         Hidden (3 neurons)    Output
  ─────         ──────────────────    ──────
  A (x₁) ─────→  H₁                  ↗
         ╲  ╱──→  H₂  ──────────────→  Ŷ
  B (x₂) ─────→  H₃                  ↗
```

**Weights:**
- `W1` shape `(2, 3)` — connects 2 inputs to 3 hidden neurons  
  (2 rows × 3 columns = 6 weights)
- `W2` shape `(3, 1)` — connects 3 hidden neurons to 1 output  
  (3 rows × 1 column = 3 weights)
- `b1` shape `(1, 3)` — one bias per hidden neuron
- `b2` shape `(1, 1)` — one bias for the output neuron

**Why small random weights?**  If all weights start at 0, every neuron  
computes the same thing and learns identically — they never differentiate.  
Small random values break this symmetry.

In [ ]:
np.random.seed(42)
W1 = np.random.randn(2, 3) * 0.5   # (inputs × hidden)
b1 = np.zeros((1, 3))              # one bias per hidden neuron
W2 = np.random.randn(3, 1) * 0.5   # (hidden × output)
b2 = np.zeros((1, 1))              # one bias for the output

print("W1 (input → hidden):")
print(W1)
print("\nW2 (hidden → output):")
print(W2)

## Forward Pass

Data flows **forward** through the network: inputs → hidden layer → output.

Each layer does two things:
1. **Linear combination**: `z = X · W + b`
2. **Activation**: `a = sigmoid(z)` — adds non-linearity

Without the sigmoid, stacking layers would just be one big linear function  
— no more expressive than a single neuron.

In [ ]:
# --- Hidden layer ---
# z1 shape: (4, 3) — one row per example, one column per hidden neuron
z1 = X @ W1 + b1      # linear combination
a1 = sigmoid(z1)      # activation — each hidden neuron's output

# --- Output layer ---
# z2 shape: (4, 1) — one prediction per example
z2 = a1 @ W2 + b2     # linear combination using hidden outputs
a2 = sigmoid(z2)      # final prediction (probability)

print("Initial predictions (before any training):")
for i, (inputs, pred, true) in enumerate(zip(X, a2, y)):
    print(f'  {inputs[0]} XOR {inputs[1]}  → {pred[0]:.3f}  (true: {true[0]})')

## Loss Function — Mean Squared Error

We use MSE here (same as linear regression) for simplicity:

```
loss = mean( (predicted − true)² )
```

In [ ]:
loss = ((a2 - y) ** 2).mean()
print(f'Initial loss: {loss:.4f}')
print('(0 would be perfect, 0.25 is terrible for this problem)')

## Backward Pass — Backpropagation

This is where the magic happens.  We apply the **chain rule** of calculus  
to figure out how each weight contributed to the error.

We work **backwards** from the output:

```
Loss → output neuron → hidden neurons → input weights
```

### Output layer gradients

**Step 1 — How does loss change with output activation `a2`?**
```python
d_a2 = 2 * (a2 - y) / n      # derivative of MSE
```

**Step 2 — How does loss change with the pre-activation `z2`?**  
We multiply by the sigmoid's derivative (chain rule through sigmoid):
```python
d_z2 = d_a2 * sigmoid_d(z2)
```

**Step 3 — How does loss change with weight matrix `W2`?**  
Each weight in W2 is multiplied by a hidden activation in the forward pass,  
so we reverse that with a transpose:
```python
d_W2 = a1.T @ d_z2
```

**Step 4 — How does loss change with bias `b2`?**  
The bias adds to every example equally, so we just sum the error signal:
```python
d_b2 = d_z2.sum(axis=0, keepdims=True)
```

### Hidden layer gradients

**Step 5 — Pass the error signal back through W2 to reach hidden neurons:**
```python
d_a1 = d_z2 @ W2.T
```

**Step 6 — Chain rule through the hidden sigmoid:**
```python
d_z1 = d_a1 * sigmoid_d(z1)
```

**Steps 7 & 8 — Gradients for W1 and b1 (same pattern as output layer):**
```python
d_W1 = X.T @ d_z1
d_b1 = d_z1.sum(axis=0, keepdims=True)
```

## Training Loop — 5000 Steps

Now we run forward + backward pass 5000 times, updating weights each time.

In [ ]:
# Re-initialise weights
np.random.seed(42)
W1 = np.random.randn(2, 3) * 0.5
b1 = np.zeros((1, 3))
W2 = np.random.randn(3, 1) * 0.5
b2 = np.zeros((1, 1))

learning_rate = 0.5
losses = []
n = len(X)

for step in range(5000):
    # --- Forward pass ---
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)

    # --- Loss ---
    loss = ((a2 - y) ** 2).mean()
    losses.append(loss)

    # --- Backward pass ---
    # Output layer
    d_a2 = 2 * (a2 - y) / n          # gradient of loss w.r.t. a2
    d_z2 = d_a2 * sigmoid_d(z2)      # chain rule through sigmoid
    d_W2 = a1.T @ d_z2               # gradient for output weights
    d_b2 = d_z2.sum(axis=0, keepdims=True)   # gradient for output bias

    # Hidden layer
    d_a1 = d_z2 @ W2.T               # error signal flowing back through W2
    d_z1 = d_a1 * sigmoid_d(z1)      # chain rule through hidden sigmoid
    d_W1 = X.T @ d_z1               # gradient for hidden weights
    d_b1 = d_z1.sum(axis=0, keepdims=True)   # gradient for hidden bias

    # --- Update weights ---
    W2 -= learning_rate * d_W2
    b2 -= learning_rate * d_b2
    W1 -= learning_rate * d_W1
    b1 -= learning_rate * d_b1

print(f'Final loss: {losses[-1]:.6f}')

In [ ]:
print('XOR predictions after training:')
for inputs, pred, true in zip(X, a2, y):
    print(f'  {inputs[0]} XOR {inputs[1]}  → {pred[0]:.3f}  (expected {true[0]})')

plt.figure(figsize=(8, 3))
plt.plot(losses, color="purple")
plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.title("Neural network learning XOR")
plt.tight_layout()
plt.show()

## Visualising the Decision Boundary

Unlike the single perceptron, the network can form a **curved** boundary  
that correctly separates the XOR classes.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200),
                     np.linspace(-0.3, 1.3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

# Forward pass on the grid
g_z1 = grid @ W1 + b1
g_a1 = sigmoid(g_z1)
g_z2 = g_a1 @ W2 + b2
g_probs = sigmoid(g_z2).reshape(xx.shape)

plt.figure(figsize=(5, 5))
plt.contourf(xx, yy, g_probs, levels=50, cmap="RdYlGn", alpha=0.7)
plt.colorbar(label="P(XOR = 1)")

y_flat = y.flatten()
for (a, b_pt), label in zip(X, y_flat):
    color = "green" if label == 1 else "red"
    marker = "o" if label == 1 else "s"
    plt.scatter(a, b_pt, c=color, marker=marker, s=150,
               edgecolors="black", linewidths=1.5, zorder=5)

plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xticks([0, 1]); plt.yticks([0, 1])
plt.xlabel("Input A"); plt.ylabel("Input B")
plt.title("Neural network decision boundary — XOR solved!")
plt.tight_layout()
plt.show()

## Key Lessons

1. **Hidden layers** allow the network to learn non-linear decision boundaries.
2. **Backpropagation** is just the chain rule applied layer by layer, backwards.
3. Every gradient tells a weight: 'if you increase, does the loss go up or down?'
4. Writing this by hand reveals exactly what PyTorch `loss.backward()` computes  
   automatically — next notebook!